# Slow-convergence histograms in the unity family

This notebook is the slower companion to `reports/slow-convergence.md`.

The narrow question is useful: **how much of the sampled square settles quickly, how much gets pushed into a long tail, and how does that tail change as the power rises in `z^n - 1`?**


## 1. Why a histogram helps

A basin plot tells us **where** points go.
It is weaker at telling us how much of the square settles fast, late, or not at all under a fixed cutoff.

For the unity family, Newton's update is

$$N_n(z) = z - rac{z^n - 1}{n z^{n-1}} = rac{n-1}{n} z + rac{1}{n z^{n-1}}.$$

That last term explains why the center and the basin filaments can keep a long tail of slow starts.
The histogram turns that visual impression into a count.


In [ ]:
from newton_fractal_lab.core import iteration_histogram

histograms = [iteration_histogram(power, width=120, height=120, max_iter=40) for power in [3, 6, 9, 12]]
for histogram in histograms:
    unresolved = (histogram.stalled_count + histogram.unresolved_count) / histogram.total_points
    print(histogram.power, histogram.tail_fraction(20), unresolved)


Even before plotting anything, the tail fraction already says something structural.
If higher powers put more mass into late iterations, then the basin geometry is not just getting prettier. It is getting harder for a fixed budget to resolve.


## 2. Exact bins versus cumulative stories

Two summaries are natural here:

- **exact histogram:** how many starts converge in exactly `k` steps
- **cumulative view:** how much of the square has converged by step `k`

This repo ships the exact histogram because it exposes the long tail directly.
The cumulative view is still useful for quick checks.


In [ ]:
for histogram in histograms:
    print('power', histogram.power)
    for step in [5, 10, 20, 30, 40]:
        print('  by', step, 'steps:', round(histogram.cumulative_converged_fraction(step), 4))


## 3. What survives the comparison

Three claims usually survive a family-level pass:

1. low powers still have slow boundary starts, but most of the sampled square settles early
2. higher powers move more mass into the late tail
3. the unresolved bar at a fixed cutoff is a warning about finite-budget reading, not a proof of true divergence

That last point matters. A point missing the current cutoff is not the same thing as a point with no root basin.


## 4. A compact bucket view

Sometimes exact bins are too fine.
A coarser public reading is still helpful:

- 0-4 steps: fast settle
- 5-9 steps: middle settle
- 10-19 steps: slow settle
- 20+ steps: late tail
- unresolved: missed the current cutoff


In [ ]:
def bucket_fraction(histogram, left, right):
    return sum(histogram.converged_counts[left:right]) / histogram.total_points

for histogram in histograms:
    unresolved = (histogram.stalled_count + histogram.unresolved_count) / histogram.total_points
    print('z^{} - 1'.format(histogram.power))
    print('  fast', round(bucket_fraction(histogram, 0, 5), 4))
    print('  middle', round(bucket_fraction(histogram, 5, 10), 4))
    print('  slow', round(bucket_fraction(histogram, 10, 20), 4))
    print('  late', round(bucket_fraction(histogram, 20, histogram.max_iter + 1), 4))
    print('  unresolved', round(unresolved, 4))


## 5. Caveats

This notebook is intentionally bounded.

- the histogram depends on the sampled square `[-1.6, 1.6]^2`
- it depends on the finite cutoff `max_iter = 40`
- exact bin heights are numerical summaries, not invariant dynamical objects
- a heavy tail says the budget is under stress, not that the method has become meaningless


## 6. Problems and references

Try these next:

1. rerun the histograms at two cutoffs and ask which powers gain the most from a longer budget
2. compare the histogram tail against the radius-band scan to see whether the same powers look hard for the same reasons
3. build the same histogram for one asymmetric polynomial and check whether the tail shifts without radial symmetry

Useful references:

- the repo's own `reports/slow-convergence.md` and `reports/critical-structure.md`
- the rendered artifact `art/slow-convergence-histograms.svg`
- any complex dynamics text covering Newton maps, basin boundaries, and critical sets
